# 实验 06 – ⼼灵遥感查询

本笔记本按照课程要求完成：
- 解释 `WHERE discounted_price > 100` 为什么失败。
- 使用 `HAVING` 返回国家与平均消费。
- 演示 SQL 注入与参数化修复。
- 结尾执行 `conn.close()` 。

作业背景节选：实验 06 – ⼼灵遥感查询 主题： 不要把数据带到代码。把代码送到数据那⾥。 ⽬标： 掌握执⾏顺序（WHERE vs HAVING）并永远治愈 "SQL 注⼊ " 。 1. 利害关系（⽹络瓶颈） 想象你的数据库在弗吉尼亚（ AWS us-east-1 ），你的笔记本在伦敦的咖啡馆⾥。 表格⼤⼩为 100 GB。 今天，我们将数据库视为远程引擎，⽽不仅仅是⽂件存储系统。 第 1 部分：设置（迷你服务器） 我们今天不需要安装 Postgres 。我们将使⽤ sqlite3，它内置于每个 Python 安装中。它是⼀ 个活在⽂件中的真实 SQL 引擎。 任务： 创建两个表：users（ 10,000 ⾏）和 orders（ 100,000 ⾏）。 Python ⽅式： 通过 wifi 下载 100 GB 。在 RAM 中过滤。（时间： 5 ⼩时 + 崩溃）。 SQL ⽅式： 发送⼀个 50 字节的⽂本字符串（ "SELECT..." ）。服务器在弗吉尼亚过滤。它返 回 10 ⾏。（时间： 0.1 秒）。 import sqlite3 import pandas as pd import numpy as np import ra...

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import random
from pathlib import Path

# Reproducible random data
random.seed(42)
np.random.seed(42)

# Keep DB in the same folder as this notebook
db_path = Path('ecommerce.db')
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f'数据库文件： {db_path.resolve()}')

数据库文件： D:\date analysis\2026-04-14\ecommerce.db


In [2]:
# Create tables
cursor.execute('''
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    name TEXT,
    country TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    amount REAL
)
''')

# Populate only when empty (idempotent notebook runs)
user_count = cursor.execute('SELECT COUNT(*) FROM users').fetchone()[0]
order_count = cursor.execute('SELECT COUNT(*) FROM orders').fetchone()[0]

if user_count == 0:
    users_data = [
        (i, f'User_{i}', random.choice(['USA', 'UK', 'CA', 'DE']))
        for i in range(10000)
    ]
    cursor.executemany('INSERT INTO users (user_id, name, country) VALUES (?, ?, ?)', users_data)

if order_count == 0:
    orders_data = [
        (i, random.randint(0, 9999), round(random.uniform(10, 500), 2))
        for i in range(100000)
    ]
    cursor.executemany('INSERT INTO orders (order_id, user_id, amount) VALUES (?, ?, ?)', orders_data)

conn.commit()

print('users 行数：', cursor.execute('SELECT COUNT(*) FROM users').fetchone()[0])
print('orders 行数：', cursor.execute('SELECT COUNT(*) FROM orders').fetchone()[0])

users 行数： 10000
orders 行数： 100000


## 第2部分：执行顺序（WHERE vs SELECT）

第 2 部分： " 思维 " 顺序（执⾏ vs 书写） 你这样写 SQL ： SELECT → FROM → WHERE 数据库这样执⾏： FROM（找到桌⼦） → WHERE（过滤堆） → SELECT（展⽰纸张）。 陷阱： 尝试按 " 别名 " （新名称）过滤。 错误查询： # 检查当前计数以避免重新运行时插入重复数据 current_count = cursor.execute("SELECT count(*) FROM users").fetchone()[0] if current_count ==0: print(" 正在填充数据库 ... （这需要 10 秒） ") # 批量插入用户 users_data =[(i,f"User_{i}", random.choice(['USA','UK','CA','DE'])) for i inrange(10000)] cursor.executemany("INSERT INTO users VALUES (?,?,?)", users_data) # 批量插入订单 orders_data =[(i, random.randint(0,9999),round(random.uniform(10, 500),2))for i inrange(100000)] cursor.executemany("INSERT INTO orders VALUES (?,?,?)", orders_data) conn.commit() print(" 数据库就绪。 ") else: print(" 数据库已填充。 ") query_bad =""" SELECT amount * 0.9 as discounted_price FROM orders WHERE discounted_pric...

为什么失败： WHERE 在 SELECT 之前运⾏。数据库还不知道 discounted_price 是什么， 因为它还没运⾏ SELECT ⾏。 修复： 你必须在原始数学上过滤，或使⽤包装器（⼦查询 /CTE ）。 在你的笔记本中写出修正后的查询。 第 3 部分：聚合（ Where vs Having ） 这是数据科学中最常⻅的⾯试问题。 "WHERE 和 HAVING 的区别是什么？ " 任务： 找到所有平均每⽤⼾消费 > $250的国家。 第 1 步：错误⽅式（逻辑错误） 这将崩溃（在 Where 中误⽤聚合）。 第 2 步：正确⽅式 使⽤ HAVING 编写查询： except Exception as e: print(f" 崩溃：{e}") WHERE 过滤⾏（分组前）。 HAVING 过滤组（分组后）。 SELECT country,...

In [3]:
query_bad = '''
SELECT amount * 0.9 AS discounted_price
FROM orders
WHERE discounted_price > 100
'''

try:
    pd.read_sql(query_bad, conn)
except Exception as e:
    print('预期报错：', e)

In [4]:
# Fix 1: use raw expression inside WHERE
query_fixed_expr = '''
SELECT amount * 0.9 AS discounted_price
FROM orders
WHERE amount * 0.9 > 100
LIMIT 10
'''

df_fixed_expr = pd.read_sql(query_fixed_expr, conn)
df_fixed_expr

,discounted_price
0,156.537
1,369.189
2,416.565
3,112.905
4,364.293
5,407.808
6,161.514
7,427.905
8,335.025
9,400.392


In [5]:
# Fix 2: produce alias in subquery, then filter outside
query_fixed_subquery = '''
SELECT discounted_price
FROM (
    SELECT amount * 0.9 AS discounted_price
    FROM orders
) t
WHERE discounted_price > 100
LIMIT 10
'''

pd.read_sql(query_fixed_subquery, conn)

,discounted_price
0,156.537
1,369.189
2,416.565
3,112.905
4,364.293
5,407.808
6,161.514
7,427.905
8,335.025
9,400.392


## 第3部分：聚合过滤（WHERE vs HAVING）

为什么失败： WHERE 在 SELECT 之前运⾏。数据库还不知道 discounted_price 是什么， 因为它还没运⾏ SELECT ⾏。 修复： 你必须在原始数学上过滤，或使⽤包装器（⼦查询 /CTE ）。 在你的笔记本中写出修正后的查询。 第 3 部分：聚合（ Where vs Having ） 这是数据科学中最常⻅的⾯试问题。 "WHERE 和 HAVING 的区别是什么？ " 任务： 找到所有平均每⽤⼾消费 > $250的国家。 第 1 步：错误⽅式（逻辑错误） 这将崩溃（在 Where 中误⽤聚合）。 第 2 步：正确⽅式 使⽤ HAVING 编写查询： except Exception as e: print(f" 崩溃：{e}") WHERE 过滤⾏（分组前）。 HAVING 过滤组（分组后）。 SELECT country,AVG(amount) FROM users u JOIN orders o ON u.user_id = o.user_id WHEREAVG(amount)>250-- 这里错误 GROUPBY country query_good =""" SELECT u.country, AVG(o.amount) as avg_spend FROM users u JOIN orders o ON u.user_id = o.user_id GROUP BY u.country HAVING avg_spend > 250 """ df = pd.read_sql(query_good, conn) display(df) 3 / 5

In [6]:
query_wrong_agg = '''
SELECT u.country, AVG(o.amount) AS avg_spend
FROM users u
JOIN orders o ON u.user_id = o.user_id
WHERE AVG(o.amount) > 250
GROUP BY u.country
'''

try:
    pd.read_sql(query_wrong_agg, conn)
except Exception as e:
    print('预期报错（WHERE中使用聚合）：', e)

预期报错（WHERE中使用聚合）： Execution failed on sql '
SELECT u.country, AVG(o.amount) AS avg_spend
FROM users u
JOIN orders o ON u.user_id = o.user_id
WHERE AVG(o.amount) > 250
GROUP BY u.country
': misuse of aggregate: AVG()


In [7]:
query_having = '''
SELECT u.country, ROUND(AVG(o.amount), 2) AS avg_spend
FROM users u
JOIN orders o ON u.user_id = o.user_id
GROUP BY u.country
HAVING AVG(o.amount) > 250
ORDER BY avg_spend DESC
'''

df_having = pd.read_sql(query_having, conn)
df_having

,country,avg_spend
0,DE,255.66
1,USA,254.33
2,CA,254.23
3,UK,253.96


## 第4部分：SQL注入安全演示

观察： 数据库引擎物理上： 第 4 部分：安全违规（⼩鲍⽐表） 在业界，如果你像下⾯⽰例那样写 SQL ，你会⽴即被解雇。 这叫做SQL 注⼊。 ⿊客攻击： 想象⼀个登录界⾯。代码获取⽤⼾输⼊的字符串并直接粘贴到查询中。 教训： 永远不要将字符串连接到 SQL 中。 你必须使⽤参数化查询。数据库引擎将输⼊视为⽂本，⽽不是可执⾏代码。 修复： 1. 连接表。 2. 将它们分组为 4 个桶（ USA, UK, CA, DE ）。 3. 计算平均值。 4. 然后扔掉⼩于 250 的桶。 # 有漏洞的函数 defget_user_bad(input_name): # 危险： F- 字符串注入 query =f"SELECT * FROM users WHERE name = '{input_name}'" print(f" 正在运行：{query}") return pd.read_sql(query, conn) # 正常使用 print(get_user_bad("User_50")) # 攻击 # 黑客输入： User_50' OR '1'='1 # 结果：他们转储了 ** 整个 ** 数据库。 print(get_user_bad("User_50' OR '1'='1")) defget_user_good(input_name): # '?' 是占位符。驱动程序处理安全性。 query ="SELECT * FROM users WHERE name = ?" # 注意： pd.read_sql params 参数 return pd.read_sql(query, conn, params=(input_name,)) 4 / 5

In [8]:
def get_user_bad(input_name: str) -> pd.DataFrame:
    # Vulnerable: direct string interpolation
    query = f"SELECT user_id, name, country FROM users WHERE name = '{input_name}'"
    print('漏洞查询：', query)
    return pd.read_sql(query, conn)

normal_bad = get_user_bad('User_50')
print('正常输入返回行数：', len(normal_bad))
normal_bad.head()

漏洞查询： SELECT user_id, name, country FROM users WHERE name = 'User_50'
正常输入返回行数： 1


,user_id,name,country
0,50,User_50,DE


In [9]:
payload = "User_50' OR '1'='1"
attack_bad = get_user_bad(payload)

print('注入输入返回行数：', len(attack_bad))
attack_bad.head()

漏洞查询： SELECT user_id, name, country FROM users WHERE name = 'User_50' OR '1'='1'
注入输入返回行数： 10000


,user_id,name,country
0,0,User_0,USA
1,1,User_1,USA
2,2,User_2,CA
3,3,User_3,UK
4,4,User_4,UK


In [10]:
def get_user_good(input_name: str) -> pd.DataFrame:
    # Safe: parameterized query
    query = 'SELECT user_id, name, country FROM users WHERE name = ?'
    return pd.read_sql(query, conn, params=(input_name,))

attack_good = get_user_good("User_50' OR '1'='1")
print('修复后相同输入返回行数：', len(attack_good))
attack_good

修复后相同输入返回行数： 0


,user_id,name,country


## AI对话记录（提交项）

更新 _AI 集成作业与批判框架 致全体学⽣： 在过去两年中，计算机科学领域发⽣了根本性转变。像 ChatGPT 、 Claude 和 Kimi 这样的⼤型语⾔模型 （ LLM ）现在可以在⼏秒钟内⽣成功能代码并解决标准学术问题。 作为你们的⽼师，虽然我允许，并⿎励你们使⽤ LLM ，但是你们第⼀次的作业表现实在糟糕。 因此，从今以后，我们的评分范式正在改变。你们现在是⾼级⼯程师。 AI 是你们的初级开发者。 你们将不 再仅根据最终答案评分，⽽是根据你们如何管理、批判和纠正你们的 AI ⼯具来评分。 作业如何进⾏：迭代批判框架 对于所有未来的问题集，你们被允许（且被⿎励）使⽤ AI ⼯具。然⽽，你们不能简单地复制粘贴答案。你们 的提交必须展⽰你们的批判性思维过程。 对于每项作业，你们必须提交三个组成部分： 新的评分标准 由于 AI 可以在⼏秒钟内⽣成 " 正确 " 答案，最终可运⾏的代码不再是你们提交中最有价值的部分。你们的成绩 将按以下权重分配： 1. 最终解决⽅案（代码 / 答案） 你们的最终、可运⾏的提交。它必须是精⼼打磨、功能完整且优化的。 2. AI 对话记录 与 AI （ Deepseek 、 Kimi 等）完整对话的共享链接或 PDF 导出。这必须包括你们的初始提⽰和所有后续的 来回迭代。 3. " ⾼级⼯程师评审 " （元反思） ⼀份 1-2 ⻚的⽂档，在其中你们批判 AI 的表现。你们必须回答： 问题表述： 你们如何将总体的问题分解为提⽰？为什么你们以那种⽅式构建请求？ 批判： AI 在哪⾥失败了？它是否产⽣了幻觉？它是否使⽤了低效的数据结构？它是否错过了关键的 边界情况？指出 AI 错误或次优的具体代码⾏或逻辑。 迭代： 你们如何修正 AI 的错误？你们是否调整了提⽰策略，还是必须⼿动介⼊并重写逻辑？为什 么？ 30% - 问题表述与提⽰策略： 你们是否逻辑地处理问题？你们是否将复杂系统分解为可管理的组件，⽽不是要求 AI 在⼀个零样本提⽰ 中 " 全部完成 " ？ 50% - 批判性分析与迭代（核⼼成绩）： 你们审计 AI 的能⼒如何？发现细微错误、识别本可实现 O(N log N) 却⽤了 O(N²) 的低效之处，并解释为什 么AI 会犯错的学⽣将获得最⾼分。如果 AI 犯了错误⽽你们盲⽬提交，你们将失去这些分数。 20% - 最终解决⽅案有效性： 你们的最终提交是否真的有效？它是否满⾜作业的所有约束？

## 高级工程师评审（元反思）

关于学术诚信的最后说明 使⽤ AI 辅助思考现在是必须的。然⽽，未经核实的复制粘贴是新的抄袭。 如果你们提交的解决⽅案包含幻 觉、不存在的库，或你们⽆法解释的明显逻辑缺陷，这表明你们放弃了作为⾼级⼯程师的⻆⾊。你们将被相 应评分。 我们的⽬标是使你们在 AI 驱动的世界中不可或缺。通过强迫你们批判机器，你们将⽐以往更深⼊地掌握底层 的概念。 我期待看到你们的⼯程领导⼒。

## 补充：“把代码送到数据那里”的性能对比

本节对比两种方式：
1. 在 SQL 端完成聚合，只返回小结果集。
2. 全量读取到 Pandas 后再聚合。

In [11]:
import time

sql_query = '''
SELECT u.country, AVG(o.amount) AS avg_spend
FROM users u
JOIN orders o ON u.user_id = o.user_id
GROUP BY u.country
HAVING AVG(o.amount) > 250
'''

# Method 1: SQL pushdown
t0 = time.perf_counter()
df_sql_pushdown = pd.read_sql(sql_query, conn)
t1 = time.perf_counter()

# Method 2: full pull then local pandas processing
t2 = time.perf_counter()
users_all = pd.read_sql('SELECT user_id, country FROM users', conn)
orders_all = pd.read_sql('SELECT user_id, amount FROM orders', conn)
merged = users_all.merge(orders_all, on='user_id', how='inner')
df_local = (
    merged.groupby('country', as_index=False)['amount']
    .mean()
    .rename(columns={'amount': 'avg_spend'})
)
df_local = df_local[df_local['avg_spend'] > 250]
t3 = time.perf_counter()

benchmark = pd.DataFrame([
    {'method': 'SQL pushdown', 'seconds': round(t1 - t0, 4), 'rows_returned': len(df_sql_pushdown)},
    {'method': 'Pandas full pull', 'seconds': round(t3 - t2, 4), 'rows_returned': len(df_local)}
])

benchmark

,method,seconds,rows_returned
0,SQL pushdown,0.0527,4
1,Pandas full pull,0.1033,4


结论：本地 SQLite 场景下时间差距可能不大，但在远程数据库场景中，SQL 下推通常能明显降低网络传输和内存压力。

In [12]:
# Resource hygiene (required by assignment)
conn.close()
print('连接已关闭。')

连接已关闭。
